## Autor: Luis Alberto Marín Soto
## Fecha: 01/03/2025

In [1]:
# ———Librerías utilizadas———
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.select import Select
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Web Scraping con Selenium: Datos de la Bolsa de Valores de Lima (BVL)

Este proyecto tiene como finalidad **extraer** datos de los estados financieros de las empresas deseadas, en este caso, **Estado de Situación Financiera** (ESF), **Estado de Resultado** (ER) y **Estado de Flujos de Efectivo** (EFE). Se ha automatizado el proceso para realizar dicho fin mediante el paquete *Selenium*. La frecuencia escogida fue de periodicidad **anual**.

## Funciones generadas

A continuación, se ha definido ciertas **funciones** para reducir la cantidad de contenido en cada celda ya que estos se repetirán.

In [2]:
def refresh_dropdown(): # Se genera esta función para hacer frente a la constante actualización del DOM
    select_years = driver.find_element(By.XPATH, "//*[@id='page-view']/main/section/bvl-issuer-details/div/div/bvl-tabs/bvl-tab[5]/div/div[2]/bvl-resources/bvl-toolbar/div/div[3]/select")
    years = select_years.find_elements(By.TAG_NAME, "option")
    list_years = []
    for year in years:
        list_years.append(year.text)
    dropdown_select = Select(select_years)
    return dropdown_select, list_years

In [3]:
def accounts(): # Se genera esta función para elegir las cuentas (mediante su código) a extraer
    selected_accounts = []
    selected_accounts.append((input('Escribe el código de las cuentas que deseas extraer: ')).upper())
    while True:
        aditional_accounts = (input('Deseas añadir algún código más? (Caso contrario, escribir "Terminar": )')).upper()
        if aditional_accounts == 'TERMINAR':
            print('Has escogido los siguientes códigos: ', selected_accounts)
            print('='*80)
            break
        selected_accounts.append(aditional_accounts)
    return selected_accounts

In [4]:
# Extracción compuesta de datos: Estado de Situación Financiera
def scrap_esf():
    fecha_esf = []
    cuenta_esf = []
    selected_accounts = accounts()
    dropdown_select, list_years = refresh_dropdown()
    for i in list_years:
        try:
            dropdown_select.select_by_visible_text(f'{i}')
            time.sleep(5)
            driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", driver.find_element(By.XPATH, "//a[text()='Estado de Situación Financiera']"))
            time.sleep(2)
            driver.find_element(By.XPATH, "//a[text()='Estado de Situación Financiera']").click()
            time.sleep(1)
            fec = driver.find_element(By.XPATH, "//td[contains(text(),' BALANCE SHEET') or contains(text(),'ESTADO DE SITUACION FINANCIERA')]/following-sibling::td[1]").text
            fecha_esf.append(fec)
            cue = []
            for account in selected_accounts:
                cue.append(driver.find_element(By.XPATH, f"//td[text()='{account}']/following-sibling::td[2]").text)
            cuenta_esf.append(cue)
            print(f'{fec} -> S/. {cue}')
            time.sleep(2)
            driver.find_element(By.XPATH, "//button[@class = 'shared-modal-close']").click()
        except NoSuchElementException:
            continue
    print(f'{"="*80}\nScrapeo Finalizado!')
    return fecha_esf, cuenta_esf, selected_accounts

In [5]:
# Extracción compuesta de datos: Estado de Resultados
def scrap_er():
    fecha_er = []
    cuenta_er = []
    selected_accounts = accounts()
    dropdown_select, list_years = refresh_dropdown()
    for i in list_years:
        try:
            dropdown_select.select_by_visible_text(f'{i}')
            time.sleep(5)
            driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", driver.find_element(By.XPATH, "//a[text()='Estado de Ganancias y P¿rdidas' or text()='Estado de Resultados']"))
            time.sleep(2)
            driver.find_element(By.XPATH, "//a[text()='Estado de Ganancias y P¿rdidas' or text()='Estado de Resultados']").click()
            time.sleep(1)
            fec = driver.find_element(By.XPATH, "//td[contains(text(),'INCOME STATEMENT')]/following-sibling::td[1]").text
            fecha_er.append(fec)
            cue = []
            for account in selected_accounts:
                cue.append(driver.find_element(By.XPATH, f"//td[text()='{account}']/following-sibling::td[2]").text)
            cuenta_er.append(cue)
            print(f'{fec} -> S/. {cue}')
            time.sleep(2)
            driver.find_element(By.XPATH, "//button[@class = 'shared-modal-close']").click()
        except NoSuchElementException:
            continue
    print(f'{"="*80}\nScrapeo Finalizado!')
    return fecha_er, cuenta_er, selected_accounts

In [ ]:
# Extracción compuesta de datos: Estado de Flujos de Efectivo
def scrap_efe():
    fecha_efe = []
    cuenta_efe = []
    selected_accounts = accounts()
    dropdown_select, list_years = refresh_dropdown()
    for i in list_years[0:7]:
        try:
            dropdown_select.select_by_visible_text(f'{i}')
            time.sleep(5)
            driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", driver.find_element(By.XPATH, "//a[text()='Estado de Flujos de Efectivo']"))
            time.sleep(5)
            driver.find_element(By.XPATH, "//a[text()='Estado de Flujos de Efectivo']").click()
            time.sleep(3)
            fec = driver.find_element(By.XPATH, "//td[contains(text(),'ESTADO DE FLUJOS DE EFECTIVO')]/following-sibling::td[1]").text
            fecha_efe.append(fec)
            cue = []
            for account in selected_accounts:
                cue.append(driver.find_element(By.XPATH, f"//td[text()='{account}']/following-sibling::td[2]").text)
            cuenta_efe.append(cue)
            print(f'{fec} -> S/. {cue}')
            time.sleep(2)
            driver.execute_script("arguments[0].click();", driver.find_element(By.XPATH, "/html/body/bvl-shared-modal/div/div/div/div[1]/button"))
            #driver.find_element(By.XPATH, "/html/body/bvl-shared-modal/div/div/div/div[1]/button").click()
        except NoSuchElementException:
            continue
    print(f'{"="*80}\nScrapeo Finalizado!')
    return fecha_efe, cuenta_efe, selected_accounts

In [6]:
# Se genera esta función para almacenar los resultados en un dataframe y transformarlos para posteriormente exportarlos en formato Excel
def generate_data(Fecha, Cuenta):
    compounded_data_pt1 = pd.DataFrame(Fecha, columns = ['Fecha'])
    compounded_data_pt2 = pd.DataFrame(Cuenta, columns = selected_accounts)
    compounded_data_pt1['Fecha'] = pd.to_datetime(compounded_data_pt1['Fecha'], dayfirst=True)
    for account in selected_accounts:
        compounded_data_pt2[account] = compounded_data_pt2[account].str.replace(',', '').astype(float)
    compounded_data = pd.concat([compounded_data_pt1, compounded_data_pt2], axis = 1)
    compounded_data.rename(columns = codname, inplace = True)
    sort_date = input('Quieres que la fecha sea ordenada desde la más antigua hasta la más reciente? (Yes/No): ').lower()
    if sort_date == 'yes':
        compounded_data = compounded_data.sort_values('Fecha').reset_index(drop=True)
    return compounded_data

In [7]:
# Esta función simplemente exportará tus datos con la cabecera de tu dataframe y sin indice
def export():
    filename = input('Dale un nombre a tu archivo:')
    print(f'En unos instantes se exportará el archivo "{filename}" en formato excel')
    Historical_data.to_excel(f'{filename}.xlsx', header = True, index= False)
    print('Datos exportados!')
    return

## Hora de la Extracción!

Aquí estará el código a ejecutar para **extraer** datos desde la Bolsa de Valores de Lima (BVL). Como ya fue mencionado al inicio, la periodicidad ya está establecida (anual), lo que falta sería señalar que estado financiero se desea extraer: Estado de Situación Financiera (**ESF**) o Estado de Resultado (**ER**), y también qué cuentas se van a escoger (Estas serán elegidas mediante su **código**). Para ello, se pedirá al usuario mediante *input* que ingrese los valores deseados. Tener en cuenta que se puede elegir las cuentas que se desee.  
Ejemplo:
- Para ALICORP S.A.A.  
- https://www.bvl.com.pe/emisores/detalle?companyCode=21400 → nro_empresa = 21400
- Estando dentro de Estado de Resultados → ER
- 2D01ST	Ingresos de Actividades Ordinarias	6,685,002	7,319,192 → Código de cuenta = 2D01ST

In [9]:
nro_empresa = input('Ingresa el número de la empresa: ')
print('Se escogió la empresa nro: ', nro_empresa)
print('='*80)
driver = webdriver.Chrome()
driver.get(f"https://www.bvl.com.pe/emisores/detalle?companyCode={nro_empresa}")
element_1 = WebDriverWait(driver, 30).until(
    EC.presence_of_element_located((By.XPATH, "/html/body/bvl-root/bvl-main-layout/bvl-header/div/div[2]/div/div/div[1]/div[1]/a"))
)
time.sleep(3)
driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});",
                      driver.find_element(By.XPATH, "//*[@id='page-view']/main/section/bvl-issuer-details/div/div/bvl-tabs/ul/li[5]/a"))
time.sleep(1)
driver.find_element(By.XPATH, "//*[@id='page-view']/main/section/bvl-issuer-details/div/div/bvl-tabs/ul/li[5]/a").click()
time.sleep(1)
while True:
    rpta = str.lower(input('Presiona F12 y llama a la consola de navegador... ya lo hiciste? (Yes/No): '))
    if rpta == 'yes':
        print('='*80)
        break
    else:
        print('Te esperamos!')
# ——————SELECCIÓN DE PERIODO: AUDITORIA ANUAL——————"
time.sleep(2)
select_period = driver.find_element(By.XPATH, "/html/body/bvl-root/bvl-main-layout/bvl-page/div[2]/main/section/bvl-issuer-details/div/div/bvl-tabs/bvl-tab[5]/div/div[2]/bvl-resources/bvl-toolbar/div/div[2]/select")
periods = select_period.find_elements(By.TAG_NAME, "option")
list_period = []
for period in periods:
    list_period.append(period.text)
dropdown_selectperiod = Select(select_period)
dropdown_selectperiod.select_by_visible_text('Auditada Anual')
time.sleep(1)
# ——————SELECCIÓN DE ESTADO FINANCIERO A EXTRAER——————
while True:
    EF_selected = input('Estado de Situación Financiera -> ESF\nEstado de Resultados -> ER\nCual deseas extraer: ')
    fecha = []
    cuenta = []
    selected_accounts = []
    if EF_selected == 'ESF':
        print('Se extraerán los datos del Estado de Situación Financiera')
        print('='*80)
        fecha, cuenta, selected_accounts = scrap_esf()
        break
    if EF_selected == 'ER':
        print('Se extraerán los datos del Estado de Resultados')
        print('='*80)
        fecha, cuenta, selected_accounts = scrap_er()
        break
    if EF_selected == 'EFE':
        print('Se extraerán los datos del Estado de Flujos de Efectivo')
        print('='*80)
        fecha, cuenta, selected_accounts = scrap_efe()
        break
    else:
        print('Elige un Estado Financiero')
time.sleep(1)
driver.quit()

Ingresa el número de la empresa:  21400


Se escogió la empresa nro:  21400


Presiona F12 y llama a la consola de navegador... ya lo hiciste? (Yes/No):  Yes


Estado de Situación Financiera -> ESF
Estado de Resultados -> ER
Cual deseas extraer:  ER


Se extraerán los datos del Estado de Resultados


Escribe el código de las cuentas que deseas extraer:  2D01ST
Deseas añadir algún código más? (Caso contrario, escribir "Terminar": ) 2D02ST
Deseas añadir algún código más? (Caso contrario, escribir "Terminar": ) 2D03ST
Deseas añadir algún código más? (Caso contrario, escribir "Terminar": ) 2D04ST
Deseas añadir algún código más? (Caso contrario, escribir "Terminar": ) 2D07ST
Deseas añadir algún código más? (Caso contrario, escribir "Terminar": ) Terminar


Has escogido los siguientes códigos:  ['2D01ST', '2D02ST', '2D03ST', '2D04ST', '2D07ST']
31/12/2023 -> S/. ['6,685,002', '1,591,929', '523,062', '290,606', '188,369']
31/12/2022 -> S/. ['7,333,173', '1,338,149', '338,601', '564,595', '524,143']
31/12/2021 -> S/. ['6,478,035', '1,243,984', '260,764', '-10,512', '-33,990']
31/12/2020 -> S/. ['5,273,500', '1,321,652', '401,575', '390,085', '327,393']
31/12/2019 -> S/. ['4,687,530.00', '1,328,870.00', '550,994.00', '581,894.00', '476,228.00']
31/12/2018 -> S/. ['4,354,489.00', '1,212,348.00', '497,087.00', '585,172.00', '455,028.00']
31/12/2017 -> S/. ['4,230,270.00', '1,408,179.00', '479,802.00', '570,761.00', '453,095.00']
31/12/2016 -> S/. ['4,040,731.00', '1,257,045.00', '413,661.00', '408,547.00', '302,491.00']
31/12/2015 -> S/. ['3,913,878.00', '1,085,147.00', '290,888.00', '179,338.00', '153,588.00']
31/12/2014 -> S/. ['3,926,549.00', '1,041,782.00', '141,726.00', '28,277.00', '10,421.00']
31/12/2013 -> S/. ['3,838,726.00', '1,007,5

Tiempo de ejecución: 262 segundos

## Obtención de las cabeceras de las cuentas

In [10]:
ultimoaño = str(input('Ingresa el último año de los estados financieros publicados: '))
driver = webdriver.Chrome()
driver.get(f"https://www.bvl.com.pe/emisores/detalle?companyCode={nro_empresa}")
element_1 = WebDriverWait(driver, 30).until(
    EC.presence_of_element_located((By.XPATH, "/html/body/bvl-root/bvl-main-layout/bvl-header/div/div[2]/div/div/div[1]/div[1]/a"))
)
time.sleep(3)
driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});",
                      driver.find_element(By.XPATH, "//*[@id='page-view']/main/section/bvl-issuer-details/div/div/bvl-tabs/ul/li[5]/a"))
time.sleep(1)
driver.find_element(By.XPATH, "//*[@id='page-view']/main/section/bvl-issuer-details/div/div/bvl-tabs/ul/li[5]/a").click()
time.sleep(1)
while True:
    rpta = str.lower(input('Presiona F12 y llama a la consola de navegador... ya lo hiciste? (Yes/No): '))
    if rpta == 'yes':
        print('='*80)
        break
    else:
        print('Te esperamos!')
# ——————SELECCIÓN DE PERIODO: AUDITORIA ANUAL——————"
time.sleep(5)
select_period = driver.find_element(By.XPATH, "/html/body/bvl-root/bvl-main-layout/bvl-page/div[2]/main/section/bvl-issuer-details/div/div/bvl-tabs/bvl-tab[5]/div/div[2]/bvl-resources/bvl-toolbar/div/div[2]/select")
periods = select_period.find_elements(By.TAG_NAME, "option")
list_period = []
for period in periods:
    list_period.append(period.text)
dropdown_selectperiod = Select(select_period)
dropdown_selectperiod.select_by_visible_text('Auditada Anual')
time.sleep(2)
select_years_2 = driver.find_element(By.XPATH, "//*[@id='page-view']/main/section/bvl-issuer-details/div/div/bvl-tabs/bvl-tab[5]/div/div[2]/bvl-resources/bvl-toolbar/div/div[3]/select")
years_2 = select_years_2.find_elements(By.TAG_NAME, "option")
dropdown_select_2 = Select(select_years_2)
time.sleep(1)
dropdown_select_2.select_by_visible_text(ultimoaño)
time.sleep(2)
if EF_selected == 'ESF':
    driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", driver.find_element(By.XPATH, "//a[text()='Estado de Situación Financiera']"))
    time.sleep(2)
    driver.find_element(By.XPATH, "//a[text()='Estado de Situación Financiera']").click()
    time.sleep(1)
    print('Se extraerán los nombres pertenecientes al Estado de Situación Financiera')
if EF_selected == 'ER':
    driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", driver.find_element(By.XPATH, "//a[text()='Estado de Ganancias y P¿rdidas' or text()='Estado de Resultados']"))
    time.sleep(2)
    driver.find_element(By.XPATH, "//a[text()='Estado de Ganancias y P¿rdidas' or text()='Estado de Resultados']").click()
    time.sleep(1)
    print('Se extraerán los nombres pertenecientes al Estado de Resultados')
size = driver.find_elements(By.XPATH, "/html/body/bvl-shared-modal/div/div/div/div[2]/bvl-dynamic-table/div[2]/table/tbody/tr")
codname = {}
for row in size:
    cod = row.find_element(By.XPATH, "./td[1]").text  # Se busca dentro de la fila
    name = row.find_element(By.XPATH, "./td[2]").text
    codname[cod] = name
print('Nombres de las cuentas extraídos!')
time.sleep(1)
driver.quit()

Ingresa el último año de los estados financieros publicados:  2023
Presiona F12 y llama a la consola de navegador... ya lo hiciste? (Yes/No):  Yes


Se extraerán los nombres pertenecientes al Estado de Resultados
Nombres de las cuentas extraídos!


Tiempo de ejecución: 55 segundos

## Generación, transformación y exportación de los datos extraídos

Esta sección se verá todo lo relacionado con los datos ya extraídos. Estos se colocaran dentro de un *dataframe* para posteriormente **transformarlos** en el formato adecuado (*object* → *float*). Finalmente, se hará la exportación de los mismos en un formato **Excel (xlsx)**.

In [11]:
Historical_data = generate_data(fecha, cuenta)
Historical_data

Quieres que la fecha sea ordenada desde la más antigua hasta la más reciente? (Yes/No):  Yes


,Fecha,Ingresos de Actividades Ordinarias,Ganancia (Pérdida) Bruta,Ganancia (Pérdida) Operativa,Ganancia (Pérdida) antes de Impuestos,Ganancia (Pérdida) Neta del Ejercicio
0,2005-12-31,1817782.0,486190.0,192258.0,140640.0,87478.0
1,2006-12-31,1983819.0,520940.0,190573.0,175873.0,112143.0
2,2007-12-31,2505425.0,655403.0,214908.0,219387.0,121015.0
3,2008-12-31,3124150.0,702910.0,229872.0,169377.0,87582.0
4,2009-12-31,3033743.0,886188.0,390595.0,377340.0,222760.0
5,2010-12-31,3221839.0,967539.0,412791.0,422646.0,285647.0
6,2011-12-31,3687483.0,957121.0,452195.0,431675.0,322510.0
7,2012-12-31,3681343.0,943900.0,416745.0,407355.0,315613.0
8,2013-12-31,3838726.0,1007554.0,421777.0,237033.0,221324.0
9,2014-12-31,3926549.0,1041782.0,141726.0,28277.0,10421.0


In [12]:
export()

Dale un nombre a tu archivo: Estado de Resultados Ejemplo


En unos instantes se exportará el archivo "Estado de Resultados Ejemplo" en formato excel
Datos exportados!


Tiempo de ejecución: 23 segundos

En resumen, podemos decir que todo el código se demoraría en ejecutar un total de 7 minutos para extraer 5 cuentas. Tener en cuenta que puede demorar más si se desea extraer más cuentas.

## Anexo

In [15]:
# Clickear en Estado de Ganancias y Pérdidas
# driver.find_element(By.XPATH, "//a[text()='Estado de Ganancias y P¿rdidas' or text()='Estado de Resultados']").click()

In [16]:
# Localizar y extraer Fecha
# driver.find_element(By.XPATH, "//td[contains(text(),'INCOME STATEMENT')]/following-sibling::td[1]").text

In [17]:
# Localizar y extraer Ingresos
# driver.find_element(By.XPATH, "//td[contains(text(),'2D01ST')]/following-sibling::td[2]").text

In [18]:
# Salir del Estado de Ganancias y Pérdidas
# driver.find_element(By.XPATH, "//button[@class = 'shared-modal-close']").click()